# ETL Silver - IT Support

**Camada Silver**: Dados limpos, enriquecidos e prontos para análise.

Este notebook realiza as seguintes transformações:
1. **silver_tickets_full** — Join de `tb_tickets` + `tb_agents` + `tb_sla` com campos calculados (tempo de resolução, flags de SLA)
2. **silver_comments_enriched** — Join de `tb_tickets_comments` + `tb_tickets` com contexto do ticket

**Source/Target:** Configurados via `00_Config` (catalog e schema)

In [0]:
# --------------------------------------------------
# Configuração do ETL (herda catalog/schema do 00_Config)
# --------------------------------------------------
# MAGIC %run ./00_Config

In [0]:
%sql
-- --------------------------------------------------
-- Visualização das tabelas fonte (Bronze/Clean)
-- --------------------------------------------------
SELECT 'tb_tickets' AS tabela, COUNT(*) AS registros FROM tb_tickets
UNION ALL
SELECT 'tb_agents', COUNT(*) FROM tb_agents
UNION ALL
SELECT 'tb_sla', COUNT(*) FROM tb_sla
UNION ALL
SELECT 'tb_tickets_comments', COUNT(*) FROM tb_tickets_comments;

In [0]:
%sql
-- --------------------------------------------------
-- silver_tickets_full
-- Join: tickets + agents + sla com campos calculados
-- --------------------------------------------------
CREATE OR REPLACE TABLE silver_tickets_full AS
SELECT
    t.ticket_id,
    t.status,
    t.priority,
    t.source,
    t.topic,
    t.created_time,
    t.close_time,
    t.product_group,
    t.support_level,
    t.country,
    t.latitude,
    t.longitude,

    -- Dados do atendente
    a.agent_name,
    a.agent_group,
    a.agent_interactions,

    -- Dados de SLA
    s.expected_sla_to_resolve,
    s.expected_sla_to_first_response,
    s.first_response_time,
    s.sla_for_first_response,
    s.resolution_time,
    s.sla_for_resolution,
    s.survey_results,

    -- Campos calculados
    ROUND((UNIX_TIMESTAMP(t.close_time) - UNIX_TIMESTAMP(t.created_time)) / 3600, 2) AS resolution_hours,
    ROUND((UNIX_TIMESTAMP(s.first_response_time) - UNIX_TIMESTAMP(t.created_time)) / 3600, 2) AS first_response_hours,
    CASE WHEN t.close_time <= s.expected_sla_to_resolve THEN TRUE ELSE FALSE END AS sla_resolution_met,
    CASE WHEN s.first_response_time <= s.expected_sla_to_first_response THEN TRUE ELSE FALSE END AS sla_first_response_met,
    CURRENT_TIMESTAMP() AS etl_timestamp

FROM tb_tickets t
LEFT JOIN tb_agents a ON t.ticket_id = a.ticket_id
LEFT JOIN tb_sla s ON t.ticket_id = s.ticket_id;

In [0]:
%sql
-- --------------------------------------------------
-- Preview da tabela silver_tickets_full
-- --------------------------------------------------
SELECT * FROM silver_tickets_full LIMIT 10;

In [0]:
%sql
-- --------------------------------------------------
-- silver_comments_enriched
-- Join: comments + tickets (contexto do ticket)
-- --------------------------------------------------
CREATE OR REPLACE TABLE silver_comments_enriched AS
SELECT
    c.ticket_id,
    c.comments,
    LENGTH(c.comments) AS comment_length,

    -- Contexto do ticket
    t.status,
    t.priority,
    t.topic,
    t.country,
    t.product_group,
    t.support_level,
    t.created_time,
    CURRENT_TIMESTAMP() AS etl_timestamp

FROM tb_tickets_comments c
LEFT JOIN tb_tickets t ON c.ticket_id = t.ticket_id;

In [0]:
%sql
-- --------------------------------------------------
-- Preview da tabela silver_comments_enriched
-- --------------------------------------------------
SELECT * FROM silver_comments_enriched LIMIT 10;

In [0]:
%sql
-- --------------------------------------------------
-- Validação: Contagem das tabelas Silver criadas
-- --------------------------------------------------
SELECT 'silver_tickets_full' AS tabela, COUNT(*) AS registros FROM silver_tickets_full
UNION ALL
SELECT 'silver_comments_enriched', COUNT(*) FROM silver_comments_enriched;